# Automated Factsheet Extraction Pipeline

This notebook provides a single-run cell for extracting holdings data from multiple PDF factsheets.

## Usage
1. Place your PDF factsheets in the `factsheet_archive/` directory
2. Run all cells below
3. Output will be saved in `output/` directory


In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path().resolve().parent))

from extractors.pdf_extractor import PDFExtractor
from processors.data_normalizer import DataNormalizer, DataAggregator
from processors.metadata_extractor import MetadataExtractor
from config import ARCHIVE_DIR, OUTPUT_DIR, MAPPINGS_DIR
import pandas as pd
from datetime import datetime
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [ ]:
# Initialize processors
normalizer = DataNormalizer(
    sector_mapping_path=str(MAPPINGS_DIR / "sector_mapping.json"),
    isin_mapping_path=str(MAPPINGS_DIR / "isin_mapping.json")
)
metadata_extractor = MetadataExtractor()

# Find all PDF files
pdf_files = list(ARCHIVE_DIR.glob("*.pdf"))
logger.info(f"Found {len(pdf_files)} PDF files to process")


In [ ]:
# Process all PDFs
all_holdings = []
processing_log = []

for pdf_file in pdf_files:
    try:
        logger.info(f"Processing: {pdf_file.name}")
        
        # Extract metadata
        metadata = metadata_extractor.extract_from_filename(pdf_file.name)
        content_metadata = metadata_extractor.extract_from_content(str(pdf_file))
        metadata.update({k: v for k, v in content_metadata.items() if v})
        
        # Extract holdings
        extractor = PDFExtractor(str(pdf_file))
        holdings = extractor.extract()
        
        if holdings:
            # Normalize data
            normalized_holdings = normalizer.normalize(holdings, metadata)
            all_holdings.extend(normalized_holdings)
            
            # Validate
            validation = normalizer.validate_aum(normalized_holdings)
            
            processing_log.append({
                'file': pdf_file.name,
                'status': 'success',
                'holdings_count': len(normalized_holdings),
                'validation': validation
            })
            
            logger.info(f"✓ Extracted {len(normalized_holdings)} holdings from {pdf_file.name}")
        else:
            processing_log.append({
                'file': pdf_file.name,
                'status': 'failed',
                'error': 'No holdings extracted'
            })
            logger.warning(f"✗ Failed to extract from {pdf_file.name}")
    
    except Exception as e:
        logger.error(f"Error processing {pdf_file.name}: {str(e)}")
        processing_log.append({
            'file': pdf_file.name,
            'status': 'error',
            'error': str(e)
        })

logger.info(f"\nTotal holdings extracted: {len(all_holdings)}")


In [ ]:
# Create DataFrame and export
if all_holdings:
    df = pd.DataFrame(all_holdings)
    
    # Export to CSV and JSON
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = OUTPUT_DIR / f"holdings_{timestamp}.csv"
    json_path = OUTPUT_DIR / f"holdings_{timestamp}.json"
    
    DataAggregator.export_to_csv(df, str(csv_path))
    DataAggregator.export_to_json(df, str(json_path))
    
    logger.info(f"\n✓ Exported to:\n  - {csv_path}\n  - {json_path}")
    
    # Display summary
    print("\n" + "="*60)
    print("EXTRACTION SUMMARY")
    print("="*60)
    print(f"Total Holdings: {len(df)}")
    print(f"Unique Securities: {df['security_name'].nunique()}")
    print(f"AMCs: {df['amc'].nunique()}")
    print(f"Funds: {df['fund_name'].nunique()}")
    if 'date' in df.columns:
        print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
    print("\n" + "="*60)
    
    # Display first few rows
    display(df.head(10))
else:
    logger.error("No holdings extracted from any files")


In [ ]:
# Save processing log
log_df = pd.DataFrame(processing_log)
log_path = OUTPUT_DIR / f"processing_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
log_df.to_csv(log_path, index=False)
logger.info(f"Processing log saved to {log_path}")

display(log_df)
